In [ ]:
import pandas as pd
from nba_ou.create_training_data.create_df_to_train import create_df_to_train
# Create training data up to a specific date
date_to_train = "2026-06-10"

safe_limit = "2024-10-11"  # Optional: specify start date


df_total = create_df_to_train(
    recent_limit_to_include=date_to_train, older_limit_to_include=safe_limit
)


In [ ]:
df_total['TOTAL_POINTS']


In [ ]:
older_limit_to_include = "2025-12-01"  # Optional: specify start date


df_total['GAME_DATE'] = pd.to_datetime(df_total['GAME_DATE'])
df_eval = df_total[df_total['GAME_DATE'] >= pd.to_datetime(older_limit_to_include)]

In [ ]:
df_eval

## Make Predictions and Evaluate

## Load Models and Prepare Data

In [ ]:
import pickle
import numpy as np
import pandas as pd

# Load production models
model_path_full = "/home/adrian_alvarez/Projects/NBA_over_under_predictor/models/total_points_bet_accuracy/best_xgboost_regressor_total_bet_2026-01-21_production.pkl"
model_path_important = "/home/adrian_alvarez/Projects/NBA_over_under_predictor/models/total_points_bet_accuracy/best_xgboost_regressor_total_bet_important_2026-01-21_production.pkl"


model_path_full =  "/home/adrian_alvarez/Projects/NBA_over_under_predictor/models/total_points_bet_accuracy/best_xgboost_regressor_total_bet_2026-01-21_production_till_2025.pkl"


with open(model_path_full, "rb") as f:
    model_full = pickle.load(f)
    
with open(model_path_important, "rb") as f:
    model_important = pickle.load(f)

In [ ]:
# Prepare evaluation data (similar to training process)
from nba_ou.data_processing.missing_data.handle_missing_data import apply_missing_policy, compute_and_save_train_medians

df_eval_prepared = df_eval.copy()

# Filter and clean data
df_eval_prepared = df_eval_prepared[df_eval_prepared['TOTAL_POINTS'] > 130]
df_eval_prepared.dropna(subset=['TOTAL_OVER_UNDER_LINE'], inplace=True)
df_eval_prepared = df_eval_prepared[df_eval_prepared['TOTAL_OVER_UNDER_LINE'] > 100]

print(f"Evaluation data shape: {df_eval_prepared.shape}")

In [ ]:
df_eval_prepared.fillna(0)

In [ ]:
# Drop unnecessary columns
text_columns = df_eval_prepared.select_dtypes(include=['object', 'string']).columns

cols_to_drop = ['SEASON_ID', 'GAME_ID', 'SEASON_TYPE', 'TEAM_ID_TEAM_HOME',
                'TEAM_ID_TEAM_AWAY', 'IS_OVERTIME']

cols_to_drop += [
    "MATCHUP_TEAM_HOME", "GAME_NUMBER_TEAM_HOME", "TEAM_CITY_TEAM_AWAY",
    "TEAM_ABBREVIATION_TEAM_AWAY", "TEAM_ABBREVIATION_TEAM_HOME",
    "TEAM_CITY_TEAM_HOME", "TEAM_NAME_TEAM_AWAY", "MATCHUP_TEAM_AWAY",
    'TEAM_NAME_TEAM_HOME'
]
cols_to_drop += list(text_columns)


# Keep GAME_DATE for now, we'll drop it after
game_dates = df_eval_prepared['GAME_DATE'].copy()
df_eval_prepared.drop(columns=[col for col in cols_to_drop if col in df_eval_prepared.columns], inplace=True, errors='ignore')

In [ ]:
# Handle missing data
train_medians = compute_and_save_train_medians(df_eval_prepared)

df_eval_prepared, report = apply_missing_policy(
    df_eval_prepared,
    train_medians=train_medians,
    current_total_line_col="TOTAL_OVER_UNDER_LINE",
    drop_mode="strict",
)

print(f"Data shape after handling missing data: {df_eval_prepared.shape}")

In [ ]:
# Drop GAME_DATE and prepare X, y
df_eval_prepared.drop(columns='GAME_DATE', inplace=True, errors='ignore')

X_eval = df_eval_prepared.drop(['TOTAL_POINTS'], axis=1, errors='ignore')
y_eval = df_eval_prepared['TOTAL_POINTS']

print(f"X_eval shape: {X_eval.shape}")
print(f"y_eval shape: {y_eval.shape}")

In [ ]:
# Over/Under betting accuracy function
def over_under_betting_accuracy(y_true, y_pred, betting_line):
    """
    Calculate the accuracy of over/under betting decisions.
    
    A bet is successful if:
    - We predict OVER (pred > line) and actual is OVER (true > line), OR
    - We predict UNDER (pred < line) and actual is UNDER (true < line)
    """
    pred_diff = y_pred - betting_line
    true_diff = y_true - betting_line
    
    correct_predictions = np.sign(pred_diff) == np.sign(true_diff)
    accuracy = np.mean(correct_predictions)
    
    return accuracy

In [ ]:
# Model 1: Full features
y_pred_full = model_full.predict(X_eval)
betting_line = X_eval['TOTAL_OVER_UNDER_LINE'].values

# Calculate metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error

mae_full = mean_absolute_error(y_eval, y_pred_full)
mse_full = mean_squared_error(y_eval, y_pred_full)
rmse_full = root_mean_squared_error(y_eval, y_pred_full)
ou_accuracy_full = over_under_betting_accuracy(y_eval, y_pred_full, betting_line)

print("=" * 50)
print("MODEL 1: Full Features")
print("=" * 50)
print(f"MAE: {mae_full:.4f}")
print(f"MSE: {mse_full:.4f}")
print(f"RMSE: {rmse_full:.4f}")
print(f"Over/Under Betting Accuracy: {ou_accuracy_full:.2%}")
print()

In [ ]:
# Model 2: Important features only
trained_features_important = model_important.feature_names_in_
X_eval_important = X_eval[trained_features_important]

y_pred_important = model_important.predict(X_eval_important)

mae_important = mean_absolute_error(y_eval, y_pred_important)
mse_important = mean_squared_error(y_eval, y_pred_important)
rmse_important = root_mean_squared_error(y_eval, y_pred_important)
ou_accuracy_important = over_under_betting_accuracy(y_eval, y_pred_important, betting_line)

print("=" * 50)
print("MODEL 2: Important Features Only")
print("=" * 50)
print(f"MAE: {mae_important:.4f}")
print(f"MSE: {mse_important:.4f}")
print(f"RMSE: {rmse_important:.4f}")
print(f"Over/Under Betting Accuracy: {ou_accuracy_important:.2%}")
print()

In [ ]:
# Create comparison dataframe
results_df = pd.DataFrame({
    'Game': range(len(y_eval)),
    'Actual_Total': y_eval.values,
    'Betting_Line': betting_line,
    'Pred_Full': y_pred_full,
    'Pred_Important': y_pred_important,
    'Actual_Over': (y_eval.values > betting_line).astype(int),
    'Pred_Over_Full': (y_pred_full > betting_line).astype(int),
    'Pred_Over_Important': (y_pred_important > betting_line).astype(int),
})

results_df['Correct_Full'] = (results_df['Actual_Over'] == results_df['Pred_Over_Full']).astype(int)
results_df['Correct_Important'] = (results_df['Actual_Over'] == results_df['Pred_Over_Important']).astype(int)

results_df

In [ ]:
# Summary statistics
print("=" * 50)
print("VALIDATION SUMMARY")
print("=" * 50)
print(f"Total games evaluated: {len(y_eval)}")
print(f"\nActual games that went OVER: {(y_eval.values > betting_line).sum()}")
print(f"Actual games that went UNDER: {(y_eval.values <= betting_line).sum()}")
print(f"\nModel Full - Correct predictions: {results_df['Correct_Full'].sum()}/{len(y_eval)}")
print(f"Model Important - Correct predictions: {results_df['Correct_Important'].sum()}/{len(y_eval)}")

In [ ]:
results_df.to_csv("evaluation_results_recent_games.csv", index=False)